In [ ]:
# 03_modeling_and_tuning.ipynb

import pandas as pd
import numpy as np
import joblib
import optuna

from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Load processed data
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print("Data loaded.")

# Initialize models
xgb = XGBRegressor()
lgb = LGBMRegressor()
cat = CatBoostRegressor(verbose=0)

# Train models
xgb.fit(X_train, y_train)
lgb.fit(X_train, y_train)
cat.fit(X_train, y_train)

# Evaluation function
def evaluate(model, name):
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    print(f"{name} MAE:", mae)
    return mae

# Evaluate base models
evaluate(xgb, "XGBoost")
evaluate(lgb, "LightGBM")
evaluate(cat, "CatBoost")

# Stacking model
stack = StackingRegressor(
    estimators=[
        ("xgb", xgb),
        ("lgb", lgb),
        ("cat", cat)
    ],
    final_estimator=Ridge()
)

stack.fit(X_train, y_train)

# Evaluate stacked model
pred_stack = stack.predict(X_test)
print("Stacked MAE:", mean_absolute_error(y_test, pred_stack))

# Optuna tuning (example with XGBoost)
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3)
    }

    model = XGBRegressor(**params)
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    return mean_absolute_error(y_test, pred)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print("Best parameters:", study.best_params)

# Save final model
joblib.dump(stack, "../models/final_model.pkl")
print("Model saved to ../models/final_model.pkl")